# 08. Laya 체험 — 다국어 System 1 decision engine

[Laya](https://github.com/NandhaKishorM/laya)는 Convai Innovations가 공개한 오픈소스(Apache-2.0) decision model 패밀리입니다.

- **non-autoregressive**: BERT 계열 encoder(English 421M / multilingual 322M)가 한 번의 forward pass로 choice/noul/score에 답합니다 — 지금까지 본 decision model 중 가장 작습니다 (CPU로도 실용적).
- **다국어 100+**: `Router()`가 입력 언어를 감지해 한국어 등 비영어 입력을 `laya-multilingual` checkpoint로 자동 라우팅합니다.
- RLCD(strictly proper scoring rule 기반 RL)로 학습된 **calibrated 확률** 반환.
- `laya[serve]`는 **`/v1/systemone`** (Jev 호환 wire format)도 제공하고, LangChain/LangGraph/LlamaIndex/CrewAI/MCP 통합, TypeScript 포트(`npm install laya-ts`)까지 있습니다.
- 완전 무료 self-host — API key/계정 불필요.

이 노트북은 HTTP 서버 없이 **Python native `Router` API를 kernel 안에서 직접** 사용합니다 (`pip install laya`는 agent venv에 완료, checkpoint는 /data/hf-cache에 다운로드됨).

In [1]:
import sys, os, time
sys.path.insert(0, "/home/ubuntu/LLMRouter/OpenSource/workshop")   # ws_common 경로
os.environ.setdefault("HF_HOME", "/data/hf-cache")
import ws_common as W
from laya import Router

router = Router()    # 첫 사용 시 checkpoint 다운로드
print("laya Router ready")

laya Router ready


## 1. 한국어 질문 하나로 판단 3종을 한 번에

Laya는 한 state에 여러 질문을 묶어 보낼 수 있습니다. 금융 상담 문맥으로 **업무 분류(choice) + 긴급성(noul) + 고객 감정(score)** 을 동시에 물어봅니다. `routing.model`이 `multilingual`로 찍히는지(한국어 자동 감지) 확인하세요.

In [2]:
state = "지갑을 도둑맞았어요! 카드 정지시켜 주세요. 빨리요!!"
questions = {
    "d2_team": {"type": "choice",
                "instructions": "Which banking support specialist team should handle this customer message?",
                "criteria": {c: None for c in
                             ["cards", "card_security", "disputes_fees", "transfers",
                              "accounts_balance", "top_up", "exchange_fx", "refunds",
                              "declined_pay", "identity_verif", "account_mgmt", "general_faq"]}},
    "urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "emotion": {"type": "score", "instructions": "How upset is the customer?",
                "criteria": ["calm", "worried", "very upset"]},
}

router.predict(state, questions)   # warm-up (측정 제외)
t0 = time.perf_counter()
r = router.predict(state, questions)
ms = (time.perf_counter() - t0) * 1000

print(f"checkpoint: {r['routing']['model']}   (질문 3개 합산 {ms:,.1f}ms)")
a = r["answers"]
print(f"  d2_team -> {a['d2_team']['choice']} (confidence {a['d2_team']['confidence']:.3f})")
print(f"  urgent  -> P(yes) = {a['urgent']['noul']:.3f}")
print(f"  emotion -> score {a['emotion']['score']:.2f}  (0=calm ~ 2=very upset)")

/data/venvs/agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2397.02it/s]

checkpoint: multilingual   (질문 3개 합산 23.8ms)
  d2_team -> card_security (confidence 0.737)
  urgent  -> P(yes) = 0.957
  emotion -> score 1.50  (0=calm ~ 2=very upset)


## 2. 우리 LangGraph workflow에 꽂기 — adapter 20줄

Laya는 Python 객체 API이므로, `.decide()` 서명에 맞춘 **얇은 adapter**만 쓰면 지금까지의 graph에 그대로 들어갑니다. backend 교체가 얼마나 작은 일인지 보여주는 예입니다.
> adapter에서 `model="multilingual"`을 **고정**합니다. 자동 라우팅에 맡기면 영어 섞인 긴 state(tool JSON 등)가 english checkpoint로 가면서 체크포인트 전환이 반복돼 느려지고, 긴 입력은 `max_len=8192`가 필요합니다. 또한 Laya는 **선택지 11개 이상 choice의 confidence는 uncalibrated**라고 스스로 경고합니다(D2 12지 해당) — label은 쓰되 confidence 기반 설계는 피하세요.


In [3]:
from agent.decisions import DecisionResult, _fallback

class LayaDecision:
    name = "laya"
    def __init__(self, router):
        self.router = router
    def decide(self, *, state_text, question, choices, decision_id):
        qs = {decision_id: {"type": "choice", "instructions": question,
                            "criteria": {c: None for c in choices}}}
        t0 = time.perf_counter()
        r = self.router.predict(state_text, qs, model="multilingual", max_len=8192)
        ms = (time.perf_counter() - t0) * 1000
        ans = r["answers"][decision_id]
        label, valid = ans["choice"], ans["choice"] in choices
        if not valid:
            label = _fallback(decision_id, choices)
        return DecisionResult(label=label, confidence=ans.get("confidence"),
                              latency_ms=round(ms, 2), valid=valid,
                              usage={}, cost_usd=0.0, raw=str(ans)[:200])

laya_dec = LayaDecision(router)
print("adapter ready")

adapter ready


In [4]:
from agent.graph_v2 import build_graph_v2
from agent.decisions import SGLangDecision

client, llm = W.make_clients()
graph_laya   = build_graph_v2(laya_dec, llm)
graph_sglang = build_graph_v2(SGLangDecision(port=8001), llm)
sc = W.SCENARIOS[1]
print("시나리오:", sc["title"], "|", sc["question"])

시나리오: 모호한 문의 -> clarification loop | 카드가 안 긁히는데 어떻게 해야 하나요?


In [5]:
res_laya = W.run_verbose(graph_laya, sc["question"], sc["script"], "Laya")


[Laya] 고객: 카드가 안 긁히는데 어떻게 해야 하나요?



STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (200ms, confidence 0.180)

STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> account_mgmt   (178ms, confidence 0.627)



STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (158ms, confidence 0.794)



STEP 4 ▶ ask_clarify — 되묻기: Sonnet이 질문 생성 -> 고객 답변 수신
      생성(Sonnet 4.5)   (2,976ms)
      상담 agent: 어떤 카드가 안 긁히시나요? 체크카드인지 신용카드인지, 그리고 카드 뒷면 마지막 4자리 숫자를 알려주시겠어요?
      고객: 어제부터 오프라인 매장 단말기에서만 승인이 거절돼요. 관련 정보: Visa 체크카드(끝자리 7712), 마지막 시도 2026-10-27 TechMart 강남점, 금액 45,000원입니다.

STEP 5 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (166ms, confidence 0.114)



STEP 6 ▶ ask_clarify — 되묻기: Sonnet이 질문 생성 -> 고객 답변 수신
      생성(Sonnet 4.5)   (2,138ms)
      상담 agent: 온라인 결제는 정상적으로 되시나요?
      고객: 지금 알려드릴 수 있는 정보는 그게 전부예요.

STEP 7 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (168ms, confidence 0.026)



STEP 8 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> account_policy_info   (156ms, confidence 0.916)
      tool 실행: account_policy_info   (0.0ms)
      판정[D6] -> goal_met   (158ms, confidence 0.677)



STEP 9 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(Sonnet 4.5)   (4,680ms)
      요약(첫 줄): Based on the tool result provided, I can extract the following resolution facts:

STEP 10 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> no_risk   (162ms, confidence 0.000)



STEP 11 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> clarify   (168ms, confidence 0.426)



STEP 12 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(Sonnet 4.5)   (5,218ms)
      --- 최종 응답 ---
      고객님, 도움을 드리고자 하나 추가 정보가 필요합니다.
      
      온라인 결제는 정상적으로 되시는지 확인이 필요합니다. 만약 온라인은 정상이고 오프라인 매장에서만 문제가 있다면, 카드의 마그네틱 선이나 칩 손상일 가능성이 있습니다.
      
      또한 다른 오프라인 매장에서도 시도해보셨는지 알려주시겠어요? 
      
      정확한 원인 파악을 위해 위 두 가지 정보를 알려주시면, 카드 재발급이 필요한지 또는 다른 조치가 필요한지 안내해드리겠습니다.
      
      감사합니다.

>>> [Laya] 결정 9회 합계 1,516ms | 생성 15,012ms | e2e 16,539ms


In [6]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")


[SGLang] 고객: 카드가 안 긁히는데 어떻게 해야 하나요?

STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (53ms, confidence 1.000)

STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> cards   (35ms, confidence 1.000)

STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (34ms, confidence 1.000)



STEP 4 ▶ ask_clarify — 되묻기: Sonnet이 질문 생성 -> 고객 답변 수신
      생성(Sonnet 4.5)   (2,401ms)
      상담 agent: 어떤 카드가 안 긁히시나요? 체크카드인가요, 신용카드인가요? 카드 번호 뒤 4자리를 알려주시겠어요?
      고객: 어제부터 오프라인 매장 단말기에서만 승인이 거절돼요. 관련 정보: Visa 체크카드(끝자리 7712), 마지막 시도 2026-10-27 TechMart 강남점, 금액 45,000원입니다.

STEP 5 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> info_sufficient   (38ms, confidence 1.000)

STEP 6 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> card_troubleshoot   (35ms, confidence 1.000)
      tool 실행: card_troubleshoot   (0.0ms)
      판정[D6] -> goal_met   (35ms, confidence 1.000)



STEP 7 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(Sonnet 4.5)   (3,359ms)
      요약(첫 줄): Based on the tool result, here are the key resolution facts:

STEP 8 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> no_risk   (38ms, confidence 1.000)

STEP 9 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> resolve   (36ms, confidence 1.000)



STEP 10 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(Sonnet 4.5)   (4,458ms)
      --- 최종 응답 ---
      고객님, 확인 결과 Visa 체크카드(끝자리 7712)의 오프라인 결제 문제가 해결되었습니다.
      
      시스템 점검을 완료하였으며, 현재 카드 상태가 정상으로 확인되어 오프라인 매장에서 다시 사용하실 수 있습니다.
      
      TechMart 강남점 등 오프라인 매장에서 재결제를 시도해 주시기 바랍니다. 만약 여전히 문제가 발생하신다면 참조번호 **REF-ws-SGLang-21395**와 함께 다시 연락 주시기 바랍니다.
      
      도움이 되셨기를 바라며, 추가 문의사항이 있으시면 언제든지 말씀해 주세요.

>>> [SGLang] 결정 8회 합계 303ms | 생성 10,218ms | e2e 10,529ms


In [7]:
W.compare(res_laya, res_sglang)

| 지표 | Laya | SGLang | 배수(Laya/SGLang) |
|---|---:|---:|---:|
| 결정 횟수 | 9 | 8 | |
| **결정 latency 합계** | **1,516ms** | **303ms** | **5.0x** |
| 생성(LLM) latency | 15,012ms | 10,218ms | **1.5x** |
| e2e | 16,539ms | 10,529ms | **1.6x** |

**결정 계층 속도 차이: 5.0x** (e2e 단축 6,010ms)

**결정별 latency**

| 노드 | 결정 | Laya | SGLang | 배수 | 판정 label |
|---|---|---:|---:|---:|---|
| guardrail_check | D1 | 200ms | 53ms | **3.8x** | in_scope (동일) |
| intent_router | D2 | 178ms | 35ms | **5.1x** | Laya=account_mgmt / SGLang=cards |
| clarify_gate | D5 | 158ms | 34ms | **4.7x** | need_clarification (동일) |
| clarify_gate | D5 | 166ms | 38ms | **4.4x** | Laya=need_clarification / SGLang=info_sufficient |
| clarify_gate | D5 | 168ms | 35ms | **4.8x** | Laya=need_clarification / SGLang=card_troubleshoot |
| tool_select | D3 | 156ms | 35ms | **4.4x** | Laya=account_policy_info / SGLang=goal_met |
| tool_review | D6 | 158ms | 38ms | **4.1x** | Laya=goal_met / SGLang=no_risk |
| risk_check | D7 | 162ms | 36ms | **4.6x** | Laya=no_risk / SGLang=resolve |

## 한국어 판단 품질 실측 (3 backend 비교)

이 워크샵의 한국어 160건(객관적 gold)으로 D1/D2를 전수 측정한 결과입니다 (`scripts/95_decider_quality_bench.py`, `results/summary/decider_quality_ko.json`):

| backend | D1 guardrail (3지) | D2 routing strict/lenient (12지) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**해석**: 한국어 특화(Laya multilingual)보다 **모델 용량이 품질 상한을 지배**했습니다 (8B > 2B > 0.3B). Laya 측정치는 공식 문서의 zero-shot 수치(0.362)와 일치하며, Laya는 RLCD fine-tuning으로 0.766까지 오르는 **'튜닝 전제' 모델**입니다. 반대로 latency·풋프린트는 작을수록 유리합니다 — 품질/비용/속도의 3축 trade-off를 그대로 보여주는 결과입니다.


## 3. 정리 — decision model 선택지 지도

이 폴더(40_other_deciders)까지 포함하면, 같은 금융 상담 판단을 다섯 가지 방식으로 체험했습니다.

| 방식 | 판단 주체 | 판정 latency(이 환경) | 비용 구조 | 특징 |
|---|---|---|---|---|
| Bedrock Haiku 4.5 | 범용 LLM 생성 | ~640ms | 토큰 과금 | 품질 기준선, 관리형 |
| SGLang /v1/systemone (01-05) | 서빙 중인 Qwen3-VL-8B | **~35ms** | 한계비용 ~0 | 멀티모달, 기존 서빙 재사용 |
| Bedrock Intelligent Routing (06) | 판단이 아니라 **생성 모델 선택** | 선택된 모델의 생성 시간 | $1/1,000건 + 토큰 | 생성 비용 최적화 |
| strands-decider 2B (07) | 전용 decision model | ~80ms | self-host | calibrated confidence, Strands SDK 통합 |
| Laya 322-421M (08) | 전용 decision model | **~30ms** (in-process) | self-host (CPU 가능) | 다국어 자동 라우팅, 최소 풋프린트 |

- **한국어 관점**: Laya는 multilingual checkpoint가 자동 선택되어 한국어를 1급으로 다룹니다 — 이 워크샵의 한국어 시나리오와 궁합이 가장 좋은 전용 모델입니다.
- **주의**: confidence의 의미(캘리브레이션 방식)가 vendor마다 다릅니다. threshold는 제품 간 이식되지 않으므로, 반드시 자기 데이터로 reliability를 다시 측정하세요.
- zero-shot으로 부족하면 Laya는 RLCD fine-tuning 경로(공식 노트북)도 제공합니다 — typed-decisions benchmark에서 base 0.362 -> fine-tuned 0.766.